In [20]:
import numpy as np 
import pandas as pd
import torch 
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt


In [21]:
torch.manual_seed(42)
np.random.seed(42)

In [22]:
#训练集 验证集
train_data = pd.read_csv('dl-bhpp/train.csv', sep=r'\s+', header=None)
#测试集
test_data = pd.read_csv('dl-bhpp/test.csv', sep=r'\s+', header=None)

In [23]:
train_data.shape, test_data.shape

((404, 14), (102, 13))

In [24]:
train_data.head()

,0,1,2,3,4,5,6,7,8,9,10,11,12,13
0,0.00632,18.0,2.31,0,0.538,6.575,65.2,4.0900,1,296.0,15.3,396.90,4.98,24.0
1,0.02729,0.0,7.07,0,0.469,7.185,61.1,4.9671,2,242.0,17.8,392.83,4.03,34.7
2,0.06905,0.0,2.18,0,0.458,7.147,54.2,6.0622,3,222.0,18.7,396.90,5.33,36.2
3,0.14455,12.5,7.87,0,0.524,6.172,96.1,5.9505,5,311.0,15.2,396.90,19.15,27.1
4,0.21124,12.5,7.87,0,0.524,5.631,100.0,6.0821,5,311.0,15.2,386.63,29.93,16.5


In [25]:
x_all = train_data.iloc[:, :-1].values
y_all = train_data.iloc[:, -1].values

In [26]:
x_all.shape, y_all.shape

((404, 13), (404,))

In [ ]:
X_test = test_data.values
X_train, X_val, y_train, y_val = train_test_split(
    x_all, y_all, test_size=0.2, random_state=42
)
#数据标准化
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)
X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std



In [28]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape, X_test.shape

((323, 13), (323,), (81, 13), (81,), (102, 13))

In [ ]:
#定义数据集和数据加载器
class BostonDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.FloatTensor(X)
        self.y = torch.FloatTensor(y)
    
    def __len__(self):
        return len(self.X)
    
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]
    
train_dataset = BostonDataset(X_train, y_train)
val_dataset = BostonDataset(X_val, y_val)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

In [41]:
#定义模型
class MLP(nn.Module):
    def __init__(self, input_dim):
        super(MLP, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(32, 1)
        )
    
    def forward(self, x):
        return self.model(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = MLP(X_train.shape[1]).to(device)

print(model)
print(f"\n :{device}")

MLP(
  (model): Sequential(
    (0): Linear(in_features=13, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.1, inplace=False)
    (3): Linear(in_features=64, out_features=32, bias=True)
    (4): ReLU()
    (5): Dropout(p=0.1, inplace=False)
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

 :cpu


In [42]:
#定义损失函数和优化器
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)


In [46]:
#训练模型
epochs = 2000
train_losses = []
val_losses = []
step = 0

for epoch in range(epochs):
    model.train()
    epoch_train_loss = 0 #每个epoch的训练损失
    #训练一个epoch
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        predictions = model(X_batch).squeeze()#模型输出是(batch_size, 1)，需要squeeze成(batch_size,)
        loss = criterion(predictions, y_batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_losses.append(loss.item())
        epoch_train_loss += loss.item()
        
        step += 1
        
    #验证模型
    model.eval()
    epoch_val_loss = 0
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            predictions = model(X_batch).squeeze()
            loss = criterion(predictions, y_batch)
            epoch_val_loss += loss.item()
            
    #计算平均损失        
    avg_train_loss = epoch_train_loss / len(train_loader)#每个epoch的平均训练损失
    avg_val_loss = epoch_val_loss / len(val_loader)#每个epoch的平均验证损失
    val_losses.append(avg_val_loss)
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch [{epoch + 1}/{epochs}], Step[{step}], "
              f"Train Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}")
        

Epoch [10/2000], Step[110], Train Loss: 5.7140, Val Loss: 10.3585
Epoch [20/2000], Step[220], Train Loss: 3.3730, Val Loss: 9.9871
Epoch [30/2000], Step[330], Train Loss: 3.9129, Val Loss: 8.3511
Epoch [40/2000], Step[440], Train Loss: 3.9468, Val Loss: 7.9565
Epoch [50/2000], Step[550], Train Loss: 4.3237, Val Loss: 7.7012
Epoch [60/2000], Step[660], Train Loss: 5.4507, Val Loss: 7.7392
Epoch [70/2000], Step[770], Train Loss: 3.8162, Val Loss: 7.4171
Epoch [80/2000], Step[880], Train Loss: 3.8952, Val Loss: 9.7491
Epoch [90/2000], Step[990], Train Loss: 5.3463, Val Loss: 8.1602
Epoch [100/2000], Step[1100], Train Loss: 3.6158, Val Loss: 7.9460
Epoch [110/2000], Step[1210], Train Loss: 3.7696, Val Loss: 7.3562
Epoch [120/2000], Step[1320], Train Loss: 4.5357, Val Loss: 7.5762
Epoch [130/2000], Step[1430], Train Loss: 3.4164, Val Loss: 7.4213
Epoch [140/2000], Step[1540], Train Loss: 4.5766, Val Loss: 10.5285
Epoch [150/2000], Step[1650], Train Loss: 3.5291, Val Loss: 8.4910
Epoch [160/

In [ ]:
#评估模型
model.eval()
with torch.no_grad():
    X_train_tensor = torch.FloatTensor(X_train).to(device)
    y_train_pred = model(X_train_tensor).squeeze().cpu().numpy()
    
    X_val_tensor = torch.FloatTensor(X_val).to(device)
    y_val_pred = model(X_val_tensor).squeeze().cpu().numpy()
    
train_mse = np.mean((y_train - y_train_pred) ** 2)
val_mse = np.mean((y_val - y_val_pred) ** 2)

print(f"train MSE: {train_mse:.4f},"
      f"val MSE: {val_mse:.4f}")
    

train MSE: 1.6323,val MSE: 12.0108


In [50]:
#预测测试集
model.eval()
with torch.no_grad():
    X_test_tensor = torch.FloatTensor(X_test).to(device)
    y_test_pred = model(X_test_tensor).squeeze().cpu().numpy()
#保存预测结果
submission = pd.DataFrame({
    'Id': np.arange(len(y_test_pred)),
    'Predict': y_test_pred
})
submission.to_csv('submission.csv', index=False)
print(submission.head())


   Id    Predict
0   0  28.159283
1   1  35.120689
2   2  23.088390
3   3  19.421019
4   4  22.070486
